# Notebook 12c — VLA v2 MQAR Capacity (Fixed + Gated VLA)
## All bugs fixed, Gated VLA introduced, proper Zoology-standard MQAR

### Critical bugs fixed from 12b
1. **VOCAB=512 was fatal**: loss starts at log(511)≈6.24 with 0.2% random accuracy.
   Gradient landscape is essentially flat — model can never escape random.
   Evidence: VLA_Debug_PhaseA.ipynb already diagnosed this and achieved 100%
   accuracy with VOCAB=128.
2. **BATCH=32 too small**: MQAR needs large batches (≥64) because each sample
   has different random KV pairs. Small batch → high-variance gradients → no learning.
3. **MQAR task format**: switched to Zoology-standard format with fixed sequence
   length, random noise tokens, and proper key/value separation.

### Architectural improvements — Gated VLA (VLA v2)
The core innovation: add a **learned decay gate** to the S matrix update.
- Old VLA: `S = S + e ⊗ α_n` (purely additive, no forgetting)
- **New Gated VLA**: `S = g * S + e ⊗ α_n` where `g = sigmoid(W_g(x))`

This gives VLA the best of both worlds:
- Sherman-Morrison penalty direction (unique to VLA, provably optimal)
- Selective forgetting gate (proven in DeltaNet/GLA/Mamba2)

### Models compared
| Model | Heads | d_model | Key Feature |
|---|---|---|---|
| Gated-VLA (NEW) | 8×d_h=64 | 512 | SM penalty + decay gate |
| Uniform-VLA | 8×d_h=64 | 512 | SM penalty only (no gate) |
| DeltaNet | 8×d_h=64 | 512 | Delta rule + decay gate |
| Transformer | 8×d_h=64 | 512 | Full quadratic attention (upper bound) |

n_pairs swept: 4, 8, 16, 32, 48, 64, 96 (capacity boundary = d_h = 64).
3 seeds. fp32 throughout.

## 0 · Setup

In [ ]:
import math, time, gc, json, glob
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as torch_checkpoint

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT    = Path('/kaggle/working/nb12c_gated_vla')
for sub in ['plots', 'logs']:
    (OUT/sub).mkdir(parents=True, exist_ok=True)

matplotlib.rcParams.update({
    'font.family' : 'DejaVu Serif', 'font.size' : 11,
    'axes.spines.top' : False, 'axes.spines.right' : False,
    'axes.grid' : True, 'grid.alpha' : 0.25, 'figure.dpi' : 150,
})

print(f'Device : {DEVICE}')
print(f'Torch  : {torch.__version__}')
print(f'Output : {OUT}')

# ── cap_rows recovery: survives kernel restarts ──────
def _recover_cap_rows():
    files = sorted(glob.glob(str(OUT/'logs'/'exp1_n*.csv')))
    rows = []
    for fp in files:
        rows.extend(pd.read_csv(fp).to_dict('records'))
    return rows

if 'cap_rows' not in dir() or not isinstance(cap_rows, list):
    cap_rows = _recover_cap_rows()
    if cap_rows:
        done_n = sorted(set(r['n_pairs'] for r in cap_rows))
        print(f'Recovered {len(cap_rows)} rows from disk for n_pairs={done_n}')
    else:
        print('No checkpoint files found -- starting cap_rows fresh.')

## 1 · Configuration (All Bugs Fixed)

In [ ]:
# ── FIXED CONFIG ─────────────────────────────────────────────────────────
# BUG FIX 1: VOCAB=512 was fatal (loss landscape flat at log(511)≈6.24).
# VLA_Debug_PhaseA.ipynb proved VOCAB=128 works → 100% accuracy.
# We use 128 for capacity testing (standard for synthetic MQAR probes).
VOCAB  = 128

# BUG FIX 2: BATCH=32 too small for MQAR. Need ≥64 for stable gradients.
BATCH  = 64

# ── Model dimensions ────────────────────────────────────────────────────
# All models: 8 heads × d_h=64 = d_model=512
# This gives capacity = d_h = 64 per head (theoretical MQAR limit)
H       = 8
D_H     = 64
D_MODEL = H * D_H  # = 512

# ── MQAR task ────────────────────────────────────────────────────────────
# Zoology-standard: fixed sequence length with noise padding
SEQ_LEN = 512    # fixed sequence length (Zoology standard)
N_PAIRS = [4, 8, 16, 32, 48, 64, 96]  # sweep across capacity boundary

# ── Training ────────────────────────────────────────────────────────────
SEEDS  = [42, 123, 999]
LR     = 3e-4     # slightly lower than 1e-3 — more stable for MQAR
WARMUP = 300      # longer warmup for SM inverse tracking
GRAD_CLIP = 1.0
CHECKPOINT_CHUNK = 32

def steps_for_n(n):
    """Budget training steps based on difficulty."""
    if n <= 8:   return 800
    if n <= 16:  return 1200
    if n <= 32:  return 1500
    if n <= 64:  return 2000
    return 2500

print('FIXED CONFIG (all bugs from 12b resolved):')
print(f'  VOCAB     = {VOCAB}  (was 512 → FATAL, random acc=0.2%)')
print(f'  BATCH     = {BATCH}  (was 32 → noisy gradients)')
print(f'  d_model   = {D_MODEL}  ({H} heads × d_h={D_H})')
print(f'  capacity  = d_h = {D_H} associations per head')
print(f'  SEQ_LEN   = {SEQ_LEN}')
print(f'  LR        = {LR}')
print(f'  WARMUP    = {WARMUP}')
print()
print(f'  Random baseline = 1/{VOCAB-1} = {1/(VOCAB-1):.4f}')
print(f'  Initial loss ≈ log({VOCAB-1}) = {math.log(VOCAB-1):.4f}')
print()
print(f'n_pairs sweep: {N_PAIRS}')
for n in N_PAIRS:
    marker = ' ← capacity boundary' if n == D_H else ''
    print(f'  n={n:4d}: {steps_for_n(n):,} steps{marker}')

## 2 · Fixed MQAR Task (Zoology Standard)

Key differences from the broken 12b version:
1. **Fixed sequence length** — position embeddings learn consistent patterns
2. **Noise padding** — random noise tokens fill unused positions (not separator)
3. **Proper key/value separation** — keys from `[0, VOCAB//2)`, values from `[VOCAB//2, VOCAB-1)`
4. **No key collisions** — guaranteed unique keys via `randperm`

In [ ]:
def make_mqar(B, n_pairs, vocab=VOCAB, seq_len=SEQ_LEN, device=DEVICE):
    """
    Zoology-standard MQAR task.
    
    Format: [noise... k1 v1 noise... k2 v2 ... | SEP | noise... q1 noise... q2 ...]
    Target: only the token AFTER each query key in the second half.
    
    Keys from [1, vocab//2), values from [vocab//2, vocab-1), separator = vocab-1.
    Noise tokens = 0 (dedicated noise token, never a key or value).
    """
    sep       = vocab - 1
    noise_tok = 0  # dedicated noise token
    key_lo, key_hi = 1, vocab // 2           # keys: [1, 64) for vocab=128
    val_lo, val_hi = vocab // 2, vocab - 1   # vals: [64, 127) for vocab=128
    
    n_keys_available = key_hi - key_lo  # 63 unique keys for vocab=128
    assert n_pairs <= n_keys_available, (
        f'n_pairs={n_pairs} > available keys={n_keys_available}. '
        f'Increase VOCAB or decrease n_pairs.')
    
    T = seq_len
    half = T // 2  # first half = KV store, second half = queries
    
    x = torch.full((B, T), noise_tok, dtype=torch.long, device=device)
    y = torch.full((B, T), -100, dtype=torch.long, device=device)
    
    for b in range(B):
        # Sample unique keys and random values
        keys = torch.randperm(n_keys_available, device=device)[:n_pairs] + key_lo
        vals = torch.randint(val_lo, val_hi, (n_pairs,), device=device)
        
        # Place KV pairs in first half at random positions
        # Each KV pair takes 2 slots: [key, value]
        n_kv_slots = 2 * n_pairs
        available_positions = half - 1  # reserve slot 0 or last for separator
        assert n_kv_slots <= available_positions, (
            f'Too many pairs ({n_pairs}) for half-length ({half})')
        
        # Randomly choose starting positions for KV pairs in first half
        positions = torch.randperm(available_positions // 2, device=device)[:n_pairs]
        positions = positions * 2  # even positions only, so key-value stay adjacent
        
        for i in range(n_pairs):
            pos = positions[i].item()
            x[b, pos]     = keys[i]
            x[b, pos + 1] = vals[i]
        
        # Separator
        x[b, half - 1] = sep
        
        # Place queries in second half
        # Query format: [query_key, ANSWER_SLOT, query_key, ANSWER_SLOT, ...]
        perm = torch.randperm(n_pairs, device=device)
        for i in range(n_pairs):
            qpos = half + 2 * i
            if qpos + 1 < T:
                x[b, qpos]     = keys[perm[i]]
                x[b, qpos + 1] = noise_tok  # placeholder for answer
                y[b, qpos + 1] = vals[perm[i]]  # target = correct value
    
    return x, y


# ── Sanity checks ────────────────────────────────────────────────────────
print('MQAR sanity checks (FIXED):')
for n in [4, 8, 16, 32, 48, 64]:
    x, y = make_mqar(4, n, VOCAB, SEQ_LEN)
    n_targets = (y != -100).sum().item()
    assert x.max().item() <= VOCAB - 1, f'token {x.max()} >= vocab {VOCAB}'
    assert x.min().item() >= 0, f'negative token'
    assert n_targets == 4 * n, f'expected {4*n} targets, got {n_targets}'
    # Check no key appears as value
    print(f'  n={n:3d}  T={x.shape[1]}  targets={n_targets:4d}  '
          f'max_tok={x.max().item()}  OK')

print(f'\nRandom baseline = 1/{VOCAB-1} = {1/(VOCAB-1):.4f}')
print(f'Initial loss should be ≈ log({VOCAB-1}) = {math.log(VOCAB-1):.4f}')

## 3 · Model Definitions

### Models:
1. **Gated VLA** (NEW) — Sherman-Morrison penalty + learned decay gate
2. **Uniform VLA** — Sherman-Morrison penalty only (no gate, baseline)
3. **DeltaNet** — Delta rule + decay gate (competitor baseline)
4. **Transformer** — Full quadratic attention (upper bound)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Shared utilities
# ══════════════════════════════════════════════════════════════════════════

def _split_raw(x, H):
    """Raw per-head slice: (B,T,D) -> (B,H,T,dh)"""
    B, T, D = x.shape
    dh = D // H
    return x.view(B, T, H, dh).permute(0, 2, 1, 3)

def _merge(x):
    """(B,H,T,dh) -> (B,T,D)"""
    B, H, T, dh = x.shape
    return x.permute(0, 2, 1, 3).reshape(B, T, H*dh)


# ══════════════════════════════════════════════════════════════════════════
# 1. GATED VLA — The New Architecture (VLA v2)
# ══════════════════════════════════════════════════════════════════════════
#
# Key innovation: S = g * S + e ⊗ α_n
#   - g = sigmoid(W_g(x)) is a per-head, per-timestep decay gate
#   - When g ≈ 1: behaves like original VLA (no forgetting)
#   - When g ≈ 0: aggressive forgetting (fresh state each step)
#   - The gate learns to balance memory retention vs interference
#
# This combines VLA's unique Sherman-Morrison penalty direction
# (provably optimal for tracking inverse covariance) with the
# selective forgetting mechanism proven in DeltaNet/GLA/Mamba2.

def _gated_vla_chunk_step(kf_c, Q_c, V_c, U_c, G_c, A, S, zk, I, isq,
                           t_offset, eps, period, per_eps):
    """One checkpointed chunk of Gated VLA recurrence."""
    B, H = A.shape[0], A.shape[1]
    Tc = kf_c.shape[2]
    ys = []
    for tc in range(Tc):
        t = t_offset + tc
        # ── Sherman-Morrison inverse update on A ──
        u   = U_c[:,:,tc,:] * isq
        zsm = torch.einsum('bhde,bhe->bhd', A, u)
        dlt = (1.0 + (u*zsm).sum(-1)).clamp(min=eps)
        A   = A - torch.einsum('bhd,bhe->bhde', zsm, zsm) / dlt.view(B,H,1,1)
        if (t+1) % period == 0:
            A = A + per_eps * I
        
        # ── Memory update with decay gate ──
        kn    = F.normalize(kf_c[:,:,tc,:], p=2, dim=-1)
        alpha = torch.einsum('bhde,bhe->bhd', A, kn)
        alphn = F.normalize(alpha, p=2, dim=-1)
        e     = V_c[:,:,tc,:] - torch.einsum('bhde,bhe->bhd', S, kn)
        
        # THE KEY CHANGE: gated decay on S
        gt = G_c[:,:,tc,:]  # (B, H, dh) — per-head gate
        S  = gt.unsqueeze(-1) * S + torch.einsum('bhd,bhe->bhde', e, alphn)
        
        # ── Output ──
        qt = Q_c[:,:,tc,:]
        zk = zk + kf_c[:,:,tc,:]
        yt = torch.einsum('bhde,bhe->bhd', S, qt)
        ys.append(yt / (zk*qt).sum(-1, keepdim=True).clamp(min=eps))
    return torch.stack(ys, dim=2), A, S, zk


class GatedVLA(nn.Module):
    """
    Gated Variational Linear Attention (VLA v2).
    
    Combines Sherman-Morrison inverse penalty direction with a learned
    decay gate on the memory matrix S. This enables selective forgetting
    to prevent interference when the state fills up.
    
    S_t = g_t * S_{t-1} + e_t ⊗ normalize(A_t k_t)
    where g_t = sigmoid(W_g(k_raw_t))
    """
    def __init__(self, d_model, H, lam=0.1, eps=1e-4, per_eps=1e-3, period=20,
                 chunk=None):
        super().__init__()
        assert d_model % H == 0
        self.H = H; self.dh = d_model // H
        self.lam=lam; self.eps=eps; self.per_eps=per_eps; self.period=period
        self.chunk = chunk if chunk is not None else CHECKPOINT_CHUNK
        dh = self.dh
        # Per-head projections (architecturally faithful to validated reference)
        self.Wq_w = nn.Parameter(torch.empty(H, dh, dh)); self.Wq_b = nn.Parameter(torch.zeros(H, dh))
        self.Wk_w = nn.Parameter(torch.empty(H, dh, dh)); self.Wk_b = nn.Parameter(torch.zeros(H, dh))
        self.Wv_w = nn.Parameter(torch.empty(H, dh, dh)); self.Wv_b = nn.Parameter(torch.zeros(H, dh))
        self.Wu_w = nn.Parameter(torch.empty(H, dh, dh))   # bias=False
        # NEW: per-head decay gate projection
        self.Wg_w = nn.Parameter(torch.empty(H, dh, dh)); self.Wg_b = nn.Parameter(torch.zeros(H, dh))
        for w in [self.Wq_w, self.Wk_w, self.Wv_w, self.Wu_w, self.Wg_w]:
            nn.init.kaiming_uniform_(w, a=math.sqrt(5))
        # Initialize gate bias to +2.0 so sigmoid ≈ 0.88 initially
        # (mostly retaining memory at the start, learning to forget)
        nn.init.constant_(self.Wg_b, 2.0)
        self.Wo   = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)

    def capacity(self): return self.H * self.dh

    def forward(self, x):
        B, T, D = x.shape; H, dh = self.H, self.dh
        xh  = _split_raw(x, H)  # (B,H,T,dh)
        kr  = torch.einsum('bhtd,hde->bhte', xh, self.Wk_w) + self.Wk_b.view(1,H,1,dh)
        kf  = F.elu(kr) + 1.0
        Q   = F.elu(torch.einsum('bhtd,hde->bhte', xh, self.Wq_w) + self.Wq_b.view(1,H,1,dh)) + 1.0
        V   = torch.einsum('bhtd,hde->bhte', xh, self.Wv_w) + self.Wv_b.view(1,H,1,dh)
        U   = F.normalize(torch.einsum('bhtd,hde->bhte', kr, self.Wu_w), p=2, dim=-1)
        # Decay gate: sigmoid so values in [0, 1]
        G   = torch.sigmoid(torch.einsum('bhtd,hde->bhte', xh, self.Wg_w) + self.Wg_b.view(1,H,1,dh))

        I   = torch.eye(dh, device=x.device, dtype=x.dtype)
        A   = (1/self.lam) * I.view(1,1,dh,dh).expand(B,H,-1,-1).clone()
        S   = torch.zeros(B, H, dh, dh, device=x.device, dtype=x.dtype)
        zk  = torch.zeros(B, H, dh, device=x.device, dtype=x.dtype)
        isq = 1.0 / math.sqrt(dh)

        ys = []
        for start in range(0, T, self.chunk):
            end = min(start + self.chunk, T)
            out_c, A, S, zk = torch_checkpoint.checkpoint(
                _gated_vla_chunk_step,
                kf[:,:,start:end], Q[:,:,start:end], V[:,:,start:end],
                U[:,:,start:end], G[:,:,start:end],
                A, S, zk, I, isq, start, self.eps, self.period, self.per_eps,
                use_reentrant=False)
            ys.append(out_c)
        out = torch.cat(ys, dim=2)
        return self.Wo(self.norm(_merge(out)))


# ══════════════════════════════════════════════════════════════════════════
# 2. ORIGINAL VLA (no gate — baseline comparison)
# ══════════════════════════════════════════════════════════════════════════

def _vla_chunk_step(kf_c, Q_c, V_c, U_c, A, S, zk, I, isq, t_offset, eps, period, per_eps):
    B, H = A.shape[0], A.shape[1]
    Tc = kf_c.shape[2]
    ys = []
    for tc in range(Tc):
        t = t_offset + tc
        u   = U_c[:,:,tc,:] * isq
        zsm = torch.einsum('bhde,bhe->bhd', A, u)
        dlt = (1.0 + (u*zsm).sum(-1)).clamp(min=eps)
        A   = A - torch.einsum('bhd,bhe->bhde', zsm, zsm) / dlt.view(B,H,1,1)
        if (t+1) % period == 0:
            A = A + per_eps * I
        kn    = F.normalize(kf_c[:,:,tc,:], p=2, dim=-1)
        alpha = torch.einsum('bhde,bhe->bhd', A, kn)
        alphn = F.normalize(alpha, p=2, dim=-1)
        e     = V_c[:,:,tc,:] - torch.einsum('bhde,bhe->bhd', S, kn)
        S     = S + torch.einsum('bhd,bhe->bhde', e, alphn)
        qt    = Q_c[:,:,tc,:]
        zk    = zk + kf_c[:,:,tc,:]
        yt    = torch.einsum('bhde,bhe->bhd', S, qt)
        ys.append(yt / (zk*qt).sum(-1,keepdim=True).clamp(min=eps))
    return torch.stack(ys, dim=2), A, S, zk


class UniformVLA(nn.Module):
    """Original VLA v1 — no decay gate. Baseline comparison."""
    def __init__(self, d_model, H, lam=0.1, eps=1e-4, per_eps=1e-3, period=20,
                 chunk=None):
        super().__init__()
        assert d_model % H == 0
        self.H = H; self.dh = d_model // H
        self.lam=lam; self.eps=eps; self.per_eps=per_eps; self.period=period
        self.chunk = chunk if chunk is not None else CHECKPOINT_CHUNK
        dh = self.dh
        self.Wq_w = nn.Parameter(torch.empty(H, dh, dh)); self.Wq_b = nn.Parameter(torch.zeros(H, dh))
        self.Wk_w = nn.Parameter(torch.empty(H, dh, dh)); self.Wk_b = nn.Parameter(torch.zeros(H, dh))
        self.Wv_w = nn.Parameter(torch.empty(H, dh, dh)); self.Wv_b = nn.Parameter(torch.zeros(H, dh))
        self.Wu_w = nn.Parameter(torch.empty(H, dh, dh))
        for w in [self.Wq_w, self.Wk_w, self.Wv_w, self.Wu_w]:
            nn.init.kaiming_uniform_(w, a=math.sqrt(5))
        self.Wo   = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)

    def capacity(self): return self.H * self.dh

    def forward(self, x):
        B, T, D = x.shape; H, dh = self.H, self.dh
        xh  = _split_raw(x, H)
        kr  = torch.einsum('bhtd,hde->bhte', xh, self.Wk_w) + self.Wk_b.view(1,H,1,dh)
        kf  = F.elu(kr) + 1.0
        Q   = F.elu(torch.einsum('bhtd,hde->bhte', xh, self.Wq_w) + self.Wq_b.view(1,H,1,dh)) + 1.0
        V   = torch.einsum('bhtd,hde->bhte', xh, self.Wv_w) + self.Wv_b.view(1,H,1,dh)
        U   = F.normalize(torch.einsum('bhtd,hde->bhte', kr, self.Wu_w), p=2, dim=-1)

        I   = torch.eye(dh, device=x.device, dtype=x.dtype)
        A   = (1/self.lam) * I.view(1,1,dh,dh).expand(B,H,-1,-1).clone()
        S   = torch.zeros(B, H, dh, dh, device=x.device, dtype=x.dtype)
        zk  = torch.zeros(B, H, dh, device=x.device, dtype=x.dtype)
        isq = 1.0 / math.sqrt(dh)

        ys = []
        for start in range(0, T, self.chunk):
            end = min(start + self.chunk, T)
            out_c, A, S, zk = torch_checkpoint.checkpoint(
                _vla_chunk_step,
                kf[:,:,start:end], Q[:,:,start:end], V[:,:,start:end], U[:,:,start:end],
                A, S, zk, I, isq, start, self.eps, self.period, self.per_eps,
                use_reentrant=False)
            ys.append(out_c)
        out = torch.cat(ys, dim=2)
        return self.Wo(self.norm(_merge(out)))


# ══════════════════════════════════════════════════════════════════════════
# 3. DELTANET (competitor baseline)
# ══════════════════════════════════════════════════════════════════════════

def _deltanet_chunk_step(kt_c, vt_c, qt_c, gt_c, S):
    Tc = kt_c.shape[2]
    ys = []
    for tc in range(Tc):
        kt, vt, qt, gt = kt_c[:,:,tc,:], vt_c[:,:,tc,:], qt_c[:,:,tc,:], gt_c[:,:,tc,:]
        pred = torch.einsum('bhde,bhe->bhd', S, kt)
        S = gt.unsqueeze(-1) * S + torch.einsum('bhd,bhe->bhde', vt - pred, kt)
        ys.append(torch.einsum('bhde,bhe->bhd', S, qt))
    return torch.stack(ys, dim=2), S


class DeltaNet(nn.Module):
    """DeltaNet baseline — delta rule + decay gate."""
    def __init__(self, d_model, H, eps=1e-6, chunk=None):
        super().__init__()
        self.H=H; self.dh=d_model//H; self.eps=eps
        self.chunk = chunk if chunk is not None else CHECKPOINT_CHUNK
        dh = self.dh
        self.Wq_w = nn.Parameter(torch.empty(H, dh, dh)); self.Wq_b = nn.Parameter(torch.zeros(H, dh))
        self.Wk_w = nn.Parameter(torch.empty(H, dh, dh)); self.Wk_b = nn.Parameter(torch.zeros(H, dh))
        self.Wv_w = nn.Parameter(torch.empty(H, dh, dh)); self.Wv_b = nn.Parameter(torch.zeros(H, dh))
        self.Wg_w = nn.Parameter(torch.empty(H, dh, dh)); self.Wg_b = nn.Parameter(torch.zeros(H, dh))
        for w in [self.Wq_w, self.Wk_w, self.Wv_w, self.Wg_w]:
            nn.init.kaiming_uniform_(w, a=math.sqrt(5))
        self.Wo = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, D = x.shape; H, dh = self.H, self.dh
        xh = _split_raw(x, H)
        Q  = F.elu(torch.einsum('bhtd,hde->bhte', xh, self.Wq_w) + self.Wq_b.view(1,H,1,dh)) + 1.0
        K  = F.normalize(torch.einsum('bhtd,hde->bhte', xh, self.Wk_w) + self.Wk_b.view(1,H,1,dh), p=2, dim=-1)
        V  = torch.einsum('bhtd,hde->bhte', xh, self.Wv_w) + self.Wv_b.view(1,H,1,dh)
        G  = torch.sigmoid(torch.einsum('bhtd,hde->bhte', xh, self.Wg_w) + self.Wg_b.view(1,H,1,dh))

        S = torch.zeros(B, H, dh, dh, device=x.device, dtype=x.dtype)
        ys = []
        for start in range(0, T, self.chunk):
            end = min(start + self.chunk, T)
            out_c, S = torch_checkpoint.checkpoint(
                _deltanet_chunk_step,
                K[:,:,start:end], V[:,:,start:end], Q[:,:,start:end], G[:,:,start:end], S,
                use_reentrant=False)
            ys.append(out_c)
        out = torch.cat(ys, dim=2)
        return self.Wo(_merge(out))


# ══════════════════════════════════════════════════════════════════════════
# 4. TRANSFORMER (quadratic attention — upper bound)
# ══════════════════════════════════════════════════════════════════════════

class CausalSelfAttention(nn.Module):
    """Standard causal multi-head attention (quadratic, but gold standard)."""
    def __init__(self, d_model, H):
        super().__init__()
        self.H = H; self.dh = d_model // H
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.Wo  = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        B, T, D = x.shape; H, dh = self.H, self.dh
        qkv = self.qkv(x).reshape(B, T, 3, H, dh).permute(2, 0, 3, 1, 4)
        Q, K, V = qkv[0], qkv[1], qkv[2]  # (B, H, T, dh)
        
        scale = 1.0 / math.sqrt(dh)
        attn = (Q @ K.transpose(-2, -1)) * scale  # (B, H, T, T)
        # Causal mask
        mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
        attn = attn.masked_fill(mask, float('-inf'))
        attn = F.softmax(attn, dim=-1)
        
        out = attn @ V  # (B, H, T, dh)
        return self.Wo(self.norm(_merge(out)))


# ══════════════════════════════════════════════════════════════════════════
# LM Backbone (shared by all models)
# ══════════════════════════════════════════════════════════════════════════

class Block(nn.Module):
    def __init__(self, attn, d, ff_mult=2):
        super().__init__()
        self.ln1 = nn.LayerNorm(d); self.ln2 = nn.LayerNorm(d)
        self.attn = attn
        self.ff = nn.Sequential(
            nn.Linear(d, d*ff_mult), nn.GELU(), nn.Linear(d*ff_mult, d))
    def forward(self, x):
        return x + self.ff(self.ln2(x + self.attn(self.ln1(x))))


class TinyLM(nn.Module):
    def __init__(self, attn_fn, d, vocab=VOCAB, n_layers=2):
        super().__init__()
        self.tok = nn.Embedding(vocab, d)
        self.pos = nn.Embedding(8192, d)
        self.blocks = nn.ModuleList([Block(attn_fn(d), d) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d)
        self.head = nn.Linear(d, vocab, bias=False)
        self.head.weight = self.tok.weight  # weight tying
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)

    def forward(self, idx):
        B, T = idx.shape
        x = (self.tok(idx) +
             self.pos(torch.arange(T, device=idx.device).unsqueeze(0)))
        for b in self.blocks: x = b(x)
        return self.head(self.ln_f(x))  # FIX: use self.head() not @ tok.weight.T


# ── Model Registry ──────────────────────────────────────────────────────
C = {'gated_vla': '#E17055', 'uniform_vla': '#55EFC4',
     'deltanet': '#FDCB6E', 'transformer': '#74B9FF'}
M = {'gated_vla': 's', 'uniform_vla': 'o',
     'deltanet': '^', 'transformer': 'D'}

MODEL_REGISTRY = {
    'Gated-VLA':     (lambda d: GatedVLA(d, H=H),
                      D_MODEL, C['gated_vla'], M['gated_vla']),
    'Uniform-VLA':   (lambda d: UniformVLA(d, H=H),
                      D_MODEL, C['uniform_vla'], M['uniform_vla']),
    'DeltaNet':      (lambda d: DeltaNet(d, H=H),
                      D_MODEL, C['deltanet'], M['deltanet']),
    'Transformer':   (lambda d: CausalSelfAttention(d, H=H),
                      D_MODEL, C['transformer'], M['transformer']),
}

print('NaN checks & param counts:')
for name, (factory, d_model, _, _) in MODEL_REGISTRY.items():
    m = TinyLM(factory, d_model, VOCAB).to(DEVICE)
    x = torch.randint(0, VOCAB, (2, 32), device=DEVICE)
    o = m(x)
    p = sum(q.numel() for q in m.parameters())/1e6
    print(f'  {name:14s}: NaN={torch.isnan(o).any().item()}  params={p:.2f}M  d_model={d_model}')
    del m; gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()

## 4 · Quick Smoke Test — Does Learning Happen?

Before running the full sweep, verify that at least one model escapes
random accuracy within 200 steps on n_pairs=8. If this fails, something
is still broken.

In [ ]:
def run_mqar(attn_fn, d_model, n_pairs,
             steps=2000, batch=BATCH, lr=LR, warmup=WARMUP, seed=42,
             log_every=100, verbose=True, seq_len=SEQ_LEN):
    """
    Train a TinyLM on MQAR and return final eval accuracy.
    Plain fp32 throughout — load-bearing for VLA's Sherman-Morrison.
    """
    torch.manual_seed(seed)
    model = TinyLM(attn_fn, d_model, VOCAB).to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    def lrf(s):
        if s < warmup: return s / max(warmup, 1)
        return 0.5 * (1 + math.cos(math.pi*(s-warmup)/max(steps-warmup,1)))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lrf)

    rand_baseline = 1/(VOCAB-1)
    t0 = time.time()
    model.train()
    for step in range(1, steps + 1):
        x, y = make_mqar(batch, n_pairs, VOCAB, seq_len)
        loss = F.cross_entropy(
            model(x).view(-1, VOCAB), y.view(-1), ignore_index=-100)
        if not torch.isfinite(loss):
            if verbose: print(f'      [non-finite loss at step {step} -- stopping]')
            break
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        opt.step(); sched.step()

        if verbose and (step % log_every == 0 or step == 1):
            model.eval()
            with torch.no_grad():
                pa = []
                for _ in range(4):
                    xp, yp = make_mqar(64, n_pairs, VOCAB, seq_len)
                    mp = yp != -100
                    if mp.any():
                        pa.append((model(xp).argmax(-1)[mp]==yp[mp]).float().mean().item())
                probe_acc = float(np.mean(pa)) if pa else 0.0
            model.train()
            status = '✓ LEARNING' if probe_acc > rand_baseline * 5 else '~ random'
            print(f'      step {step:5d}/{steps}  loss={loss.item():.4f}  '
                  f'acc={probe_acc:.4f}  ({time.time()-t0:.0f}s)  {status}')

    model.eval(); accs = []
    with torch.no_grad():
        for _ in range(20):
            xv, yv = make_mqar(64, n_pairs, VOCAB, seq_len)
            mask   = yv != -100
            if mask.any():
                accs.append(
                    (model(xv).argmax(-1)[mask] == yv[mask]).float().mean().item())
    final_acc = float(np.mean(accs)) if accs else 0.0
    del model; gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    return final_acc


# ── SMOKE TEST: n_pairs=8, 400 steps ──
print('='*60)
print(' SMOKE TEST: n_pairs=8, 400 steps — verify learning happens')
print('='*60)
print(f' Expected: random baseline = {1/(VOCAB-1):.4f}')
print(f' Must see acc > 0.05 within 200 steps.')
print()

for mname in ['Gated-VLA', 'DeltaNet']:
    factory, d_model, _, _ = MODEL_REGISTRY[mname]
    print(f'  [{mname}]')
    acc = run_mqar(factory, d_model, n_pairs=8, steps=400, log_every=100, seed=42)
    status = '✅ PASS' if acc > 0.5 else ('⚡ PARTIAL' if acc > 0.05 else '❌ FAIL')
    print(f'    Final acc = {acc:.4f}  {status}')
    print()

## 5 · Speed Benchmark

In [ ]:
print('Benchmarking 10 steps per model at n_pairs=16...')
print('='*60)
for mname, (factory, d_model, _, _) in MODEL_REGISTRY.items():
    torch.manual_seed(0)
    model = TinyLM(factory, d_model, VOCAB).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    x, y = make_mqar(BATCH, 16, VOCAB, SEQ_LEN)
    if DEVICE=='cuda': torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(10):
        loss = F.cross_entropy(model(x).view(-1,VOCAB), y.view(-1), ignore_index=-100)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    if DEVICE=='cuda': torch.cuda.synchronize()
    dt = (time.time()-t0)/10
    max_steps = max(steps_for_n(n) for n in N_PAIRS)
    print(f'  {mname:14s}: {dt*1000:6.0f} ms/step  '
          f'-> ~{dt*max_steps/60:.1f} min for {max_steps} steps')
    del model, opt; gc.collect()
    if DEVICE=='cuda': torch.cuda.empty_cache()
print()
print('If any model shows multi-second/step, reduce SEQ_LEN or BATCH.')

## 6 · MQAR Capacity Sweep

One cell per n_pairs value. Each saves its checkpoint immediately.
`cap_rows` recovers from disk on kernel restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Full capacity sweep — all n_pairs values
# ══════════════════════════════════════════════════════════════════════════

for n in N_PAIRS:
    # Skip if already computed
    if any(r['n_pairs'] == n for r in cap_rows):
        print(f'n_pairs={n} already computed, skipping.')
        continue
    
    st = steps_for_n(n)
    marker = ' ← CAPACITY BOUNDARY' if n == D_H else ''
    print(f'\n{"="*65}')
    print(f' n_pairs={n}   steps={st:,}{marker}')
    print(f'{"="*65}')

    t_cell = time.time()
    for mname, (factory, d_model, _, _) in MODEL_REGISTRY.items():
        accs = []
        print(f'\n  [{mname}]  d_model={d_model}')
        for seed in SEEDS:
            t0  = time.time()
            acc = run_mqar(factory, d_model, n, steps=st, batch=BATCH, lr=LR,
                           seed=seed, log_every=st//5)
            dt  = time.time() - t0
            accs.append(round(acc, 4))
            print(f'    seed={seed:4d}: acc={acc:.4f}   ({dt:.0f}s)')
        mean, std = float(np.mean(accs)), float(np.std(accs))
        print(f'    {"-"*40}')
        print(f'    MEAN={mean:.4f}  STD={std:.4f}  seeds={accs}')
        cap_rows.append({'n_pairs': n, 'model': mname,
                         'mean': round(mean,4), 'std': round(std,4),
                         'seeds': str(accs)})

    elapsed = (time.time() - t_cell) / 60
    print(f'\n  n_pairs={n} done in {elapsed:.1f} min')

    pd.DataFrame([r for r in cap_rows if r['n_pairs']==n]).to_csv(
        OUT/'logs'/f'exp1_n{n}.csv', index=False)
    print(f'  Checkpoint saved: exp1_n{n}.csv')

# ── Final aggregation ────────────────────────────────────────────────────
print(f'\n{"="*65}')
print(' ALL SWEEPS COMPLETE — building master CSV')
print(f'{"="*65}')

df_cap = pd.DataFrame(cap_rows)
df_cap.to_csv(OUT/'logs'/'exp1_capacity.csv', index=False)

print('\nFull capacity table:')
print(f'  {"n_pairs":>8}  {"Gated-VLA":>11}  {"Uniform-VLA":>13}  {"DeltaNet":>10}  {"Transformer":>13}')
print(f'  {"-"*62}')
for nv in N_PAIRS:
    row = {r['model']: r for _, r in df_cap[df_cap.n_pairs==nv].iterrows()}
    g = row.get('Gated-VLA',    {}).get('mean', float('nan'))
    u = row.get('Uniform-VLA',  {}).get('mean', float('nan'))
    d = row.get('DeltaNet',     {}).get('mean', float('nan'))
    t = row.get('Transformer',  {}).get('mean', float('nan'))
    print(f'  {nv:>8}:  {g:>11.4f}  {u:>13.4f}  {d:>10.4f}  {t:>13.4f}')

print(f'\nSaved: exp1_capacity.csv')

## 7 · Publication-Ready Figure

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── Panel (a): Capacity curve ────────────────────────────────────────────
ax = axes[0]
for mname, (factory, d_model, col, mk) in MODEL_REGISTRY.items():
    g = df_cap[df_cap.model == mname].sort_values('n_pairs')
    if g.empty: continue
    ax.errorbar(g.n_pairs, g['mean'], yerr=g['std'],
                color=col, marker=mk, lw=2.5, ms=9, capsize=5, capthick=1.5,
                label=f'{mname}')

ax.axvline(D_H, color='gray', ls='--', lw=1.8, alpha=0.6,
           label=f'd_h={D_H} boundary')
ax.axhline(0.80, color='#636e72', ls=':', lw=1.5)
ax.text(N_PAIRS[-1]*0.98, 0.82, 'pass (0.80)', ha='right', fontsize=9, color='#636e72')
ax.axhline(1/(VOCAB-1), color='lightgray', ls=':', lw=1.2)

ax.set(title=f'(a) MQAR Capacity: Gated VLA vs Baselines\n'
             f'd_model={D_MODEL}, {H} heads × d_h={D_H}',
       xlabel='n_pairs stored', ylabel='Eval accuracy (mean ± std, 3 seeds)')
ax.set_ylim(-0.02, 1.05); ax.legend(fontsize=9, loc='lower left')

# ── Panel (b): Bar chart at capacity boundary ────────────────────────────
ax2 = axes[1]
boundary_n = D_H  # at the theoretical capacity boundary
g_boundary = df_cap[df_cap.n_pairs == boundary_n]
if not g_boundary.empty:
    models_sorted = g_boundary.sort_values('mean', ascending=False)
    bars = ax2.bar(range(len(models_sorted)), models_sorted['mean'],
                   yerr=models_sorted['std'],
                   color=[C.get(n.lower().replace('-','_').replace(' ','_'), 'gray')
                          for n in models_sorted['model']],
                   alpha=0.85, edgecolor='black', linewidth=0.8,
                   capsize=5)
    for bar, (_, row) in zip(bars, models_sorted.iterrows()):
        ax2.text(bar.get_x()+bar.get_width()/2,
                 bar.get_height()+0.02,
                 f'{row["mean"]:.3f}', ha='center', fontsize=10, fontweight='bold')
    ax2.set_xticks(range(len(models_sorted)))
    ax2.set_xticklabels([n.replace('-',' ') for n in models_sorted['model']],
                         rotation=15, ha='right')
    ax2.axhline(0.80, color='#636e72', ls=':', lw=1.5)
    ax2.set(title=f'(b) Accuracy at n_pairs={boundary_n} (capacity boundary)',
            ylabel='Accuracy')
    ax2.set_ylim(0, 1.1)

plt.suptitle('Gated VLA: Combining Sherman-Morrison Penalty with Selective Forgetting',
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUT/'plots'/'paper_figure_nb12c.pdf', bbox_inches='tight')
plt.savefig(OUT/'plots'/'paper_figure_nb12c.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: paper_figure_nb12c.{pdf,png}')

## 8 · Results Summary

In [ ]:
print('='*70)
print('NOTEBOOK 12c RESULTS — GATED VLA (VLA v2)')
print('='*70)
print(f'  All models: {H} heads × d_h={D_H} = d_model={D_MODEL}')
print(f'  Capacity boundary = d_h = {D_H}')
print()

for n in N_PAIRS:
    row = {r['model']: r for _, r in df_cap[df_cap.n_pairs==n].iterrows()}
    g = row.get('Gated-VLA',   {}).get('mean', float('nan'))
    u = row.get('Uniform-VLA', {}).get('mean', float('nan'))
    d = row.get('DeltaNet',    {}).get('mean', float('nan'))
    t = row.get('Transformer', {}).get('mean', float('nan'))
    marker = ' ← CAPACITY BOUNDARY' if n == D_H else ''
    print(f'  n={n:4d}:  Gated={g:.4f}  Uniform={u:.4f}  '
          f'DeltaNet={d:.4f}  Transformer={t:.4f}{marker}')

# Key comparisons
print()
print('KEY COMPARISONS:')
for n in [D_H//2, D_H, D_H + D_H//2]:
    if n not in N_PAIRS: continue
    row = {r['model']: r for _, r in df_cap[df_cap.n_pairs==n].iterrows()}
    g = row.get('Gated-VLA',  {}).get('mean', 0)
    d = row.get('DeltaNet',   {}).get('mean', 0)
    u = row.get('Uniform-VLA',{}).get('mean', 0)
    t = row.get('Transformer',{}).get('mean', 0)
    print(f'  n={n}: Gated-VLA={g:.4f} vs DeltaNet={d:.4f} '
          f'(delta={g-d:+.4f}) vs Transformer={t:.4f}')
    if g > d:
        print(f'    → Gated-VLA WINS by {(g-d)*100:.1f}pp')
    elif g == d:
        print(f'    → TIE')
    else:
        print(f'    → DeltaNet leads by {(d-g)*100:.1f}pp')

# Save manifest
manifest = {
    'config': {
        'vocab': VOCAB, 'batch': BATCH, 'd_model': D_MODEL,
        'H': H, 'd_h': D_H, 'lr': LR, 'warmup': WARMUP,
        'seq_len': SEQ_LEN,
    },
    'capacity_results': df_cap.to_dict('records'),
    'bugs_fixed': [
        'VOCAB 512->128 (loss landscape was flat)',
        'BATCH 32->64 (noisy gradients)',
        'MQAR task format (Zoology standard)',
        'TinyLM forward head usage',
    ],
    'new_architecture': 'Gated VLA: S = g*S + e⊗α_n where g=sigmoid(W_g(x))',
}
with open(OUT/'manifest.json','w') as f:
    json.dump(manifest, f, indent=2)
print(f'\nAll files: {OUT}')